# Lab02S01 — Coleta dos 1.000 repositórios Java mais populares do GitHub

| RQ | Pergunta | Métrica de Processo | Métrica de Qualidade |
|:--:|----------|---------------------|----------------------|
| 01 | Popularidade vs qualidade? | Estrelas | CBO, DIT, LCOM |
| 02 | Maturidade vs qualidade? | Idade (anos) | CBO, DIT, LCOM |
| 03 | Atividade vs qualidade? | Nº de releases | CBO, DIT, LCOM |
| 04 | Tamanho vs qualidade? | LOC, linhas de comentário | CBO, DIT, LCOM |

## 1. Setup

In [ ]:
import requests
import json
import time
import os
from datetime import datetime, timezone
import pandas as pd

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("Cole seu GitHub Token: ").strip()

GITHUB_API = "https://api.github.com/graphql"
HEADERS = {"Authorization": f"Bearer {GITHUB_TOKEN}"}

assert GITHUB_TOKEN, "Defina seu GITHUB_TOKEN antes de continuar!"
print("Token OK")

## 2. Query GraphQL

Coletamos os top-1000 repos Java por estrelas, incluindo:
- `stargazerCount` (popularidade)
- `createdAt` (maturidade)
- `releases.totalCount` (atividade)
- `url` (para clone posterior)

In [ ]:
QUERY = """
{
  search(query: "language:Java sort:stars-desc stars:>100", type: REPOSITORY, first: 10, after: CURSOR) {
    pageInfo { hasNextPage endCursor }
    nodes {
      ... on Repository {
        nameWithOwner
        url
        stargazerCount
        createdAt
        updatedAt
        primaryLanguage { name }
        releases { totalCount }
        diskUsage
      }
    }
  }
}
"""

## 3. Função de requisição com retry

In [ ]:
MAX_RETRIES = 5

def fetch_graphql(query_str):
    for attempt in range(1, MAX_RETRIES + 1):
        r = requests.post(GITHUB_API, json={"query": query_str}, headers=HEADERS)
        if r.status_code == 200:
            data = r.json()
            if "errors" in data:
                raise RuntimeError(f"GraphQL errors: {data['errors']}")
            return data
        if r.status_code in (403, 502, 503, 504) and attempt < MAX_RETRIES:
            wait = 2 ** attempt
            print(f"  [retry {attempt}] HTTP {r.status_code} — aguardando {wait}s...")
            time.sleep(wait)
        else:
            r.raise_for_status()

## 4. Coleta dos 1.000 repositórios Java

In [ ]:
TARGET = 1000
nodes_raw = []
cursor = None

while len(nodes_raw) < TARGET:
    q = QUERY.replace('after: CURSOR', f'after: "{cursor}"' if cursor else '')
    result = fetch_graphql(q)
    page = result["data"]["search"]
    nodes_raw.extend(page["nodes"])
    print(f"  {len(nodes_raw)}/{TARGET} coletados")

    if not page["pageInfo"]["hasNextPage"]:
        break
    cursor = page["pageInfo"]["endCursor"]
    time.sleep(2)

nodes_raw = nodes_raw[:TARGET]
print(f"\nColeta finalizada: {len(nodes_raw)} reposit\u00f3rios Java")

## 5. Montar DataFrame

In [ ]:
now = datetime.now(timezone.utc)

rows = []
for r in nodes_raw:
    created = pd.to_datetime(r["createdAt"])
    idade_anos = round((now - created).days / 365.25, 2)

    rows.append({
        "nome":           r["nameWithOwner"],
        "url":            r["url"],
        "estrelas":       r["stargazerCount"],
        "criado_em":      created,
        "idade_anos":     idade_anos,
        "releases":       r["releases"]["totalCount"],
        "linguagem":      r["primaryLanguage"]["name"] if r["primaryLanguage"] else None,
        "disk_usage_kb":  r.get("diskUsage", 0),
    })

df = pd.DataFrame(rows)
df.index += 1
df.index.name = "#"

print(f"DataFrame criado: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head(10)

## 6. Visão geral dos dados

In [ ]:
print("Estat\u00edsticas b\u00e1sicas:")
df[["estrelas", "idade_anos", "releases"]].describe()

## 7. Salvar dados

In [ ]:
os.makedirs("data", exist_ok=True)

df.to_csv("data/repos_java_top1000.csv", index=False)
print("Salvo: data/repos_java_top1000.csv")

df.to_json("data/repos_java_top1000.json", orient="records", indent=2, force_ascii=False, date_format="iso")
print("Salvo: data/repos_java_top1000.json")

print(f"\nTotal de reposit\u00f3rios: {len(df)}")